# Experiment 4 — Sessions 1–4 calibration/validation (Qwen2.5-Omni-3B, IEMOCAP)

**Purpose:** independently test, on data disjoint from Session 5, whether NoHist/Hist3
agreement is a genuine inference-time reliability signal (Candidate 3 from the Experiment‑4
design) rather than a Session‑5-specific artifact. **This notebook never reads, writes, or
references Session 5 in any way.** Session 5's frozen results (NoHist macro‑F1 38.28, Hist8
42.16, Hist3 42.03) are not recomputed, not compared against inside this notebook, and not
used to choose anything here.

**Sample:** N=1,200, proportionally stratified by gold class, drawn from the 5,758-utterance
Sessions‑1‑4 six-class pool (120 dialogues) — sized by a pre-registered power analysis (see
`kaggle_prep/build_s14_calibration_sample.py` docstring), not from Session‑5 outcomes.
Validated 11/11 locally (`kaggle_prep/validate_s14_calibration.py`): identical ids/order/fields
between arms; identical 6-class label set/mapping to Session 5; zero dialogue overlap with
Session 5; Hist3 window reconstructed independently and matches the saved file byte-for-byte;
current turn never in its own window; anonymized Speaker_1/2 (no gender leakage); behavioral
leakage test confirms the window construction is independent of gold emotion/VAD columns;
`iemocap_eval_lib.py` sha256-identical to Experiments 2/3 (same parser/prompt/scoring).

**Two arms, same session, same loaded model — identical protocol to Experiments 2/3:**

| | NoHist_s14 | Hist3_s14 |
|---|---|---|
| Target audio/transcript/gold | `s14_calib_nohist.json` | identical |
| History | none | 3 preceding turns, transcript-only, anonymized `Speaker_1`/`Speaker_2`, current turn excluded |
| Model / decoding / seed | Qwen2.5-Omni-3B, greedy, seed=11 | identical |
| Parser / scoring | `iemocap_eval_lib.py` (sha256 `24fac1f49d6f049c...`, == Exp2/3's) | identical file |

**This notebook computes and prints only S1-4 results.** Any decision about a disagreement
fallback rule must be made from what this run shows — **never from Session‑5 labels or
Session‑5 correctness.**


## 0 · Kaggle setup

1. **Add data:** upload `kaggle_prep/kaggle_upload_s14calib/` as a **NEW, separate** Kaggle
   Dataset (do NOT merge with the Session-5 dataset used in Experiments 2/3). Contains
   `s14_calib_nohist.json`, `s14_calib_hist3.json`, tiny variants, `manifest_s14_calibration.json`,
   and `audio/` (1,200 wavs, ~180 MB).
2. **Accelerator:** GPU **T4 x2**. **Internet:** On. No secret required.
3. Run cells top to bottom. Smoke test (36 ids x 2 arms) first; then full run
   (1,200 ids x 2 arms — expect noticeably less than Experiment 2's ~68 min two-arm run,
   since N is 26% smaller).
4. Same persistence discipline as Experiments 2/3: periodic `results_bundle.zip` checkpoints;
   **Save Version → "Save & Run All (Commit)"** before closing the session (not Quick Save).


In [16]:
# ============================ CONFIG ============================
MODEL_ID       = "Qwen/Qwen2.5-Omni-3B"   # identical to Experiments 2/3
WINDOW         = 3                          # documentation only; already baked into s14_calib_hist3.json
CONTROL_TAG    = "s14_nohist_newparser"
TREATMENT_TAG  = "s14_hist3_newparser"

RUN_SMOKE_TEST = True
RUN_FULL       = True
FORCE_FULL     = False

LIMIT          = None
MAX_NEW_TOKENS = 12       # identical to Experiments 2/3
FORCE_FP16     = None
SEED           = 11       # identical to Experiments 2/3

OUTPUT_DIR     = "/kaggle/working/results"

EVAL_LIB_SHA256_EXPECTED = "24fac1f49d6f049c1485b4769ce523a292ae692fad4657a9fab7be6af6034298"  # == Experiments 2/3's recorded sha256
print("MODEL_ID:", MODEL_ID, "| WINDOW:", WINDOW, "| SEED:", SEED)
print("This run uses ONLY Sessions 1-4 data. Session 5 is not referenced anywhere below.")


MODEL_ID: Qwen/Qwen2.5-Omni-3B | WINDOW: 3 | SEED: 11
This run uses ONLY Sessions 1-4 data. Session 5 is not referenced anywhere below.


In [17]:
import subprocess, sys
def pip(*a): subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a], check=True)
pip("-U", "transformers>=4.52.3", "accelerate>=0.34", "qwen-omni-utils", "librosa", "soundfile")
# Pre-applied fix (needed in Experiment 2/3's Kaggle image): librosa can pull a scipy upgrade
# that becomes ABI-inconsistent with the preinstalled numpy.
pip("-U", "--force-reinstall", "numpy", "scipy")
print("pip done")

pip done


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.67.0 which is incompatible.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.22, but you have numpy 2.5.3 which is incompatible.
ydata-profiling 4.18.4 requires scipy<1.17,>=1.8, but you have scipy 1.18.1 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60

In [18]:
 #================= output-persistence safety net =================
import os, json, time, random, hashlib, base64
import numpy as np
# --- numpy/scipy private-API compat shim (Kaggle image bug, harmless) ---
# numpy >=2.4.4 dropped the private _blas_supports_fpe symbol that this scipy build
# still probes at import time (e.g. via sklearn/scipy.stats used by iemocap_eval_lib),
# raising AttributeError before any of our code runs. Restoring it as a no-op stub is
# purely cosmetic (BLAS FPE-support detection): it never touches any tensor, weight,
# or prediction, so it cannot change any eval result.
import numpy._core._multiarray_umath as _mu
if not hasattr(_mu, "_blas_supports_fpe"):
    _mu._blas_supports_fpe = lambda *a, **k: False
import torch

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

import transformers
print("torch       :", torch.__version__)
print("transformers:", transformers.__version__)
print("CUDA        :", torch.cuda.is_available(), "| GPUs:", torch.cuda.device_count(),
      "|", (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"))
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    print("compute cap :", cap)
    _is_t4 = cap[0] == 7
else:
    _is_t4 = False
USE_FP16 = _is_t4 if FORCE_FP16 is None else FORCE_FP16
DTYPE = torch.float16 if USE_FP16 else torch.bfloat16
print("dtype       :", DTYPE)
os.makedirs(OUTPUT_DIR, exist_ok=True)

import zipfile
ZIP_PATH = "/kaggle/working/results_bundle.zip"

def make_zip_bundle(note=""):
    tmp = ZIP_PATH + ".tmp"
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as zf:
        for root, _, files in os.walk(OUTPUT_DIR):
            for fn in files:
                fp = os.path.join(root, fn)
                zf.write(fp, os.path.relpath(fp, "/kaggle/working"))
    os.replace(tmp, ZIP_PATH)
    sz = os.path.getsize(ZIP_PATH)
    print(f"[checkpoint zip] {ZIP_PATH} ({sz/1e6:.2f} MB){' -- ' + note if note else ''} "
          f"-- download it now from Data > Output if you want an immediate local copy")
    return ZIP_PATH

make_zip_bundle("initial (empty) checkpoint")


torch       : 2.10.0+cu128
transformers: 5.17.0
CUDA        : True | GPUs: 2 | Tesla T4
compute cap : (7, 5)
dtype       : torch.float16
[checkpoint zip] /kaggle/working/results_bundle.zip (0.00 MB) -- initial (empty) checkpoint -- download it now from Data > Output if you want an immediate local copy


'/kaggle/working/results_bundle.zip'

In [19]:
# ---- locate the S1-4 calibration dataset (must NOT be the Session-5 dataset) ----
CANDIDATE_ROOTS = []
for base in ["/kaggle/input"]:
    if os.path.isdir(base):
        for d in sorted(os.listdir(base)):
            CANDIDATE_ROOTS.append(os.path.join(base, d))

REQUIRED_FILES = ["s14_calib_nohist.json", "s14_calib_hist3.json",
                   "s14_calib_tiny_nohist.json", "s14_calib_tiny_hist3.json"]

def _find_input_dir():
    for root in CANDIDATE_ROOTS:
        for dirpath, _, files in os.walk(root):
            if all(f in files for f in REQUIRED_FILES) and os.path.isdir(os.path.join(dirpath, "audio")):
                return dirpath
    raise FileNotFoundError(
        f"Could not find all of {REQUIRED_FILES} + audio/ under one directory in /kaggle/input. "
        f"Add the S1-4 calibration dataset (built by kaggle_prep/build_s14_calibration_sample.py "
        f"+ export_s14_calibration_audio.py). Seen roots: {CANDIDATE_ROOTS}")

INPUT_DIR = _find_input_dir()
AUDIO_DIR = os.path.join(INPUT_DIR, "audio")
print("INPUT_DIR:", INPUT_DIR)
print("has audio/:", os.path.isdir(AUDIO_DIR), "| n wav:", len(os.listdir(AUDIO_DIR)))
assert "test.json" not in os.listdir(INPUT_DIR), \
    "This looks like the Session-5 dataset, not the S1-4 calibration dataset -- STOP."


INPUT_DIR: /kaggle/input/datasets/pranavjaiganesh/kaggle-experiment4-s14-calibration/kaggle_upload_s14calib
has audio/: True | n wav: 1200


In [20]:
# ---- HF auth (optional; Qwen is ungated) ----
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle Secrets.")
except Exception as e:
    print("No HF_TOKEN secret (fine for Qwen):", e)


HF_TOKEN loaded from Kaggle Secrets.


In [21]:
# ================= iemocap_eval_lib.py, embedded verbatim as base64 =================
# UNCHANGED from Experiments 2/3 -- same file, same sha256, same parser/prompt/scoring code.
EVAL_LIB_B64 = "..."

_lib_bytes = base64.b64decode(EVAL_LIB_B64)
_lib_sha256 = hashlib.sha256(_lib_bytes).hexdigest()
print("iemocap_eval_lib.py decoded:", len(_lib_bytes), "bytes, sha256=", _lib_sha256)
assert _lib_sha256 == EVAL_LIB_SHA256_EXPECTED, (
    "iemocap_eval_lib.py content does not match Experiments 2/3's! "
    f"expected {EVAL_LIB_SHA256_EXPECTED}, got {_lib_sha256}")
print("sha256 matches Experiments 2/3's eval lib exactly -- identical parser/scoring -- OK")

with open("/kaggle/working/iemocap_eval_lib.py", "wb") as f:
    f.write(_lib_bytes)
import importlib, sys
sys.path.insert(0, "/kaggle/working")
import iemocap_eval_lib as EVAL
importlib.reload(EVAL)
print("eval lib ready. labels:", EVAL.IEMOCAP_LABELS)


iemocap_eval_lib.py decoded: 24510 bytes, sha256= 24fac1f49d6f049c1485b4769ce523a292ae692fad4657a9fab7be6af6034298
sha256 matches Experiments 2/3's eval lib exactly -- identical parser/scoring -- OK
eval lib ready. labels: ['happy', 'sad', 'neutral', 'angry', 'excited', 'frustrated']


In [22]:
# ---- load records for BOTH arms, and ASSERT pairing + label-set identity at runtime ----
def load_records(name):
    with open(os.path.join(INPUT_DIR, name), encoding="utf-8") as f:
        return json.load(f)

CONTROL_TINY = load_records("s14_calib_tiny_nohist.json")
HIST3_TINY   = load_records("s14_calib_tiny_hist3.json")
CONTROL_FULL = load_records("s14_calib_nohist.json")
HIST3_FULL   = load_records("s14_calib_hist3.json")
if LIMIT:
    CONTROL_FULL = CONTROL_FULL[:LIMIT]
    HIST3_FULL = HIST3_FULL[:LIMIT]

def assert_paired(control, treatment, tag):
    assert len(control) == len(treatment), f"{tag}: length mismatch"
    cids = [r["id"] for r in control]; tids = [r["id"] for r in treatment]
    assert cids == tids, f"{tag}: id order mismatch"
    for c, t in zip(control, treatment):
        assert c["utterance"] == t["utterance"], f"{tag}: utterance diverged for {c['id']}"
        assert c["output"] == t["output"], f"{tag}: gold label diverged for {c['id']}"
        assert c["session"] in (1, 2, 3, 4), f"{tag}: non-S1-4 session found for {c['id']}!"
        assert set(t.keys()) - set(c.keys()) == {"history_context"}
    return cids

tiny_ids = assert_paired(CONTROL_TINY, HIST3_TINY, "tiny")
full_ids = assert_paired(CONTROL_FULL, HIST3_FULL, "full")
golds_seen = {r["output"] for r in CONTROL_FULL}
assert golds_seen == set(EVAL.IEMOCAP_LABELS), f"label set mismatch: {golds_seen}"
print(f"ASSERTIONS PASSED: tiny {len(tiny_ids)} ids paired; full {len(full_ids)} ids paired; "
      f"all sessions in {{1,2,3,4}}; label set == {EVAL.IEMOCAP_LABELS}")
n_empty_hist = sum(1 for r in HIST3_FULL if r["history_context"] == "")
print(f"true dialogue-openers (empty history_context): {n_empty_hist}")


ASSERTIONS PASSED: tiny 36 ids paired; full 1200 ids paired; all sessions in {1,2,3,4}; label set == ['happy', 'sad', 'neutral', 'angry', 'excited', 'frustrated']
true dialogue-openers (empty history_context): 12


In [23]:
# ================= model loading (ONCE, shared by both arms) =================
_MODEL = {}

def load_model():
    from transformers import Qwen2_5OmniProcessor
    from qwen_omni_utils import process_mm_info
    import transformers as _tf
    OmniCls = getattr(_tf, "Qwen2_5OmniForConditionalGeneration",
                      getattr(_tf, "Qwen2_5OmniModel", None))
    if OmniCls is None:
        raise ImportError("This transformers build has no Qwen2.5-Omni class.")
    proc = Qwen2_5OmniProcessor.from_pretrained(MODEL_ID)
    # Same fix as Experiment 3: load the full model (talker included), do not pass
    # enable_audio_output=False / call disable_talker() -- this build's generate() needs
    # the talker present even though only the thinker's text output is used.
    model = OmniCls.from_pretrained(MODEL_ID, torch_dtype=DTYPE, device_map="auto")
    model.eval()
    _MODEL.update(model=model, proc=proc, process_mm_info=process_mm_info)
    print("model loaded:", MODEL_ID, "(same weights/settings as Experiments 2/3)")

load_model()


chat_template.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/667 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

[transformers] Model config: tts_text_start_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151860. This may result in unexpected behavior.
[transformers] Model config: tts_text_end_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151861. This may result in unexpected behavior.
[transformers] Model config: tts_text_pad_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151859. This may result in unexpected behavior.
[transformers] Model config: vision_start_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151652. This may result in unexpected behavior.
[transformers] Model config: vision_end_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151653. This may result in unexpected behavior.
[transformers] Model config: audio_start_token_id must be `None` or an integer within the vocabulary (between 0 and 8447

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/2543 [00:00<?, ?it/s]

[transformers] Qwen2_5OmniForConditionalGeneration LOAD REPORT from: Qwen/Qwen2.5-Omni-3B
Key                                                | Status     |  | 
---------------------------------------------------+------------+--+-
token2wav.code2wav_dit_model.rotary_embed.inv_freq | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

spk_dict.pt:   0%|          | 0.00/260k [00:00<?, ?B/s]

model loaded: Qwen/Qwen2.5-Omni-3B (same weights/settings as Experiments 2/3)


In [24]:
# ================= per-utterance inference (identical to Experiments 2/3) =================
QWEN_SYS = ("You are Qwen, a virtual human developed by the Qwen Team, Alibaba Group, "
            "capable of perceiving auditory and visual inputs, as well as generating text and speech.")

@torch.no_grad()
def infer_one(rec, history_context):
    prompt = EVAL.build_prompt(rec["utterance"], history_context)
    wav_path = os.path.join(AUDIO_DIR, rec["id"] + ".wav")
    model, proc, process_mm_info = _MODEL["model"], _MODEL["proc"], _MODEL["process_mm_info"]

    conv = [
        {"role": "system", "content": [{"type": "text", "text": QWEN_SYS}]},
        {"role": "user", "content": [
            {"type": "audio", "audio": wav_path},
            {"type": "text", "text": prompt}]},
    ]
    text = proc.apply_chat_template(conv, add_generation_prompt=True, tokenize=False)
    try:
        audios, images, videos = process_mm_info(conv, use_audio_in_video=False)
    except TypeError:
        audios, images, videos = process_mm_info(conv)
    try:
        inputs = proc(text=text, audio=audios, images=images, videos=videos,
                      return_tensors="pt", padding=True)
    except TypeError:
        inputs = proc(text=text, audios=audios, images=images, videos=videos,
                      return_tensors="pt", padding=True)
    inputs = inputs.to(model.device)
    try:
        inputs = inputs.to(model.dtype)
    except Exception:
        pass
    gkw = dict(max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    try:
        out = model.generate(**inputs, return_audio=False, **gkw)
    except TypeError:
        out = model.generate(**inputs, **gkw)
    if isinstance(out, (tuple, list)):
        out = out[0]
    gen = out[:, inputs["input_ids"].shape[1]:]
    return proc.batch_decode(gen, skip_special_tokens=True,
                             clean_up_tokenization_spaces=False)[0].strip()

t0 = time.time()
_demo_c = infer_one(CONTROL_TINY[0], None)
_demo_t = infer_one(HIST3_TINY[0], HIST3_TINY[0]["history_context"])
print(f"1st NoHist_s14 inference OK -> id={CONTROL_TINY[0]['id']} gold={CONTROL_TINY[0]['output']} raw={_demo_c!r}")
print(f"1st Hist3_s14  inference OK -> id={HIST3_TINY[0]['id']} gold={HIST3_TINY[0]['output']} raw={_demo_t!r}")
print(f"(both in {time.time()-t0:.1f}s)")


1st NoHist_s14 inference OK -> id=Ses01F_impro03_F012 gold=happy raw='Neutral. What do you think about this? Do you have any other utterances to analyze?'
1st Hist3_s14  inference OK -> id=Ses01F_impro03_F012 gold=happy raw='Neutral. What do you think about this?'
(both in 8.0s)


In [25]:
# ================= SMOKE TEST: both arms, 36 ids =================
from tqdm.auto import tqdm

def run_set(control_recs, treatment_recs, desc):
    c_raw, t_raw = [], []
    for r in tqdm(control_recs, desc=f"{desc}-nohist"):
        try: c_raw.append(infer_one(r, None))
        except Exception as e: print("nohist infer error", r["id"], repr(e)); c_raw.append("")
    for r in tqdm(treatment_recs, desc=f"{desc}-hist3"):
        try: t_raw.append(infer_one(r, r["history_context"]))
        except Exception as e: print("hist3 infer error", r["id"], repr(e)); t_raw.append("")
    return c_raw, t_raw

SMOKE_OK = None
if RUN_SMOKE_TEST:
    c_raw, t_raw = run_set(CONTROL_TINY, HIST3_TINY, "smoke")
    c_m = EVAL.score_predictions_normalized(CONTROL_TINY, c_raw)
    t_m = EVAL.score_predictions_normalized(HIST3_TINY, t_raw)
    print(f"{'id':<24}{'gold':<12}{'nohist_pred':<14}{'hist3_pred':<14}")
    for i, uid in enumerate(tiny_ids):
        cp = c_m["predictions"][i]; tp = t_m["predictions"][i]
        print(f"{uid:<24}{cp['gold']:<12}{cp['normalized_prediction']:<14}{tp['normalized_prediction']:<14}")
    n_empty_c = sum(1 for x in c_raw if not x.strip())
    n_empty_t = sum(1 for x in t_raw if not x.strip())
    SMOKE_OK = (n_empty_c <= 2) and (n_empty_t <= 2)
    print(f"\nNoHist_s14: WA={c_m['accuracy_WA']:.2f}  Hist3_s14: WA={t_m['accuracy_WA']:.2f}")
    print(f"SMOKE_OK = {SMOKE_OK}")
    make_zip_bundle("after smoke test")
else:
    print("smoke test skipped")


smoke-nohist:   0%|          | 0/36 [00:00<?, ?it/s]

smoke-hist3:   0%|          | 0/36 [00:00<?, ?it/s]

id                      gold        nohist_pred   hist3_pred    
Ses01F_impro03_F012     happy       neutral       neutral       
Ses01F_impro03_F013     happy       happy         happy         
Ses01F_impro03_M014     happy       neutral       neutral       
Ses01F_impro03_F016     happy       neutral       neutral       
Ses01F_impro03_F021     happy       neutral       neutral       
Ses01F_impro05_M002     happy       neutral       neutral       
Ses01F_impro02_M001     sad         frustrated    frustrated    
Ses01F_impro02_F002     sad         frustrated    frustrated    
Ses01F_impro02_F010     sad         neutral       frustrated    
Ses01F_impro06_F005     sad         sad           sad           
Ses01F_impro06_F013     sad         neutral       sad           
Ses01F_impro06_F025     sad         neutral       neutral       
Ses01F_impro01_F000     neutral     neutral       neutral       
Ses01F_impro02_M012     neutral     neutral       neutral       
Ses01F_impro03_M001     n

In [26]:
# ================= FULL RUN: both arms, sequential, same model =================
if RUN_FULL and RUN_SMOKE_TEST and (SMOKE_OK is False) and not FORCE_FULL:
    raise RuntimeError("Smoke test looked broken. Not starting the full run. Set FORCE_FULL=True to override.")

CONTROL_JSONL   = os.path.join(OUTPUT_DIR, f"pred_{CONTROL_TAG}.jsonl")
TREATMENT_JSONL = os.path.join(OUTPUT_DIR, f"pred_{TREATMENT_TAG}.jsonl")
ZIP_EVERY = 200

def load_done(path):
    done = {}
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    o = json.loads(line); done[o["id"]] = o["raw"]
    return done

def run_full_arm(records, jsonl_path, get_history, desc):
    done = load_done(jsonl_path)
    print(f"[{desc}] resuming: {len(done)} / {len(records)} already done")
    t0 = time.time()
    with open(jsonl_path, "a", encoding="utf-8") as f:
        for i, r in enumerate(tqdm(records, desc=desc)):
            if r["id"] in done:
                continue
            try:
                raw = infer_one(r, get_history(r))
            except Exception as e:
                print(f"[{desc}] infer error", r["id"], repr(e)); raw = ""
            done[r["id"]] = raw
            f.write(json.dumps({"id": r["id"], "raw": raw}, ensure_ascii=False) + "\n")
            f.flush()
            if (i + 1) % 100 == 0:
                el = time.time() - t0
                print(f"  [{desc}] {i+1}/{len(records)}  elapsed {el/60:.1f}m  eta {el/(i+1)*(len(records)-i-1)/60:.1f}m")
            if (i + 1) % ZIP_EVERY == 0:
                make_zip_bundle(f"mid-run checkpoint, {desc} {i+1}/{len(records)}")
    print(f"[{desc}] done in {(time.time()-t0)/60:.1f} min -> {jsonl_path}")
    make_zip_bundle(f"{desc} arm complete")
    return done

if RUN_FULL:
    control_done = run_full_arm(CONTROL_FULL, CONTROL_JSONL, lambda r: None, "nohist_s14")
    treatment_done = run_full_arm(HIST3_FULL, TREATMENT_JSONL, lambda r: r["history_context"], "hist3_s14")
else:
    print("full run skipped")


[nohist_s14] resuming: 0 / 1200 already done


nohist_s14:   0%|          | 0/1200 [00:00<?, ?it/s]

  [nohist_s14] 100/1200  elapsed 2.6m  eta 29.1m
  [nohist_s14] 200/1200  elapsed 5.2m  eta 26.1m
[checkpoint zip] /kaggle/working/results_bundle.zip (0.00 MB) -- mid-run checkpoint, nohist_s14 200/1200 -- download it now from Data > Output if you want an immediate local copy
  [nohist_s14] 300/1200  elapsed 7.9m  eta 23.7m
  [nohist_s14] 400/1200  elapsed 10.4m  eta 20.8m
[checkpoint zip] /kaggle/working/results_bundle.zip (0.00 MB) -- mid-run checkpoint, nohist_s14 400/1200 -- download it now from Data > Output if you want an immediate local copy
  [nohist_s14] 500/1200  elapsed 12.9m  eta 18.0m
  [nohist_s14] 600/1200  elapsed 15.3m  eta 15.3m
[checkpoint zip] /kaggle/working/results_bundle.zip (0.00 MB) -- mid-run checkpoint, nohist_s14 600/1200 -- download it now from Data > Output if you want an immediate local copy
  [nohist_s14] 700/1200  elapsed 17.6m  eta 12.6m
  [nohist_s14] 800/1200  elapsed 19.9m  eta 10.0m
[checkpoint zip] /kaggle/working/results_bundle.zip (0.00 MB) -- m

hist3_s14:   0%|          | 0/1200 [00:00<?, ?it/s]

  [hist3_s14] 100/1200  elapsed 2.5m  eta 27.3m
  [hist3_s14] 200/1200  elapsed 4.8m  eta 24.2m
[checkpoint zip] /kaggle/working/results_bundle.zip (0.01 MB) -- mid-run checkpoint, hist3_s14 200/1200 -- download it now from Data > Output if you want an immediate local copy
  [hist3_s14] 300/1200  elapsed 7.3m  eta 21.8m
  [hist3_s14] 400/1200  elapsed 9.5m  eta 19.0m
[checkpoint zip] /kaggle/working/results_bundle.zip (0.01 MB) -- mid-run checkpoint, hist3_s14 400/1200 -- download it now from Data > Output if you want an immediate local copy
  [hist3_s14] 500/1200  elapsed 11.9m  eta 16.7m
  [hist3_s14] 600/1200  elapsed 14.4m  eta 14.4m
[checkpoint zip] /kaggle/working/results_bundle.zip (0.01 MB) -- mid-run checkpoint, hist3_s14 600/1200 -- download it now from Data > Output if you want an immediate local copy
  [hist3_s14] 700/1200  elapsed 16.6m  eta 11.9m
  [hist3_s14] 800/1200  elapsed 19.0m  eta 9.5m
[checkpoint zip] /kaggle/working/results_bundle.zip (0.01 MB) -- mid-run checkp

In [27]:
# ================= SCORE + SAVE both arms =================
import pandas as pd

def read_jsonl(path):
    d = {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                o = json.loads(line); d[o["id"]] = o["raw"]
    return d

control_done = read_jsonl(CONTROL_JSONL)
treatment_done = read_jsonl(TREATMENT_JSONL)
c_missing = [r["id"] for r in CONTROL_FULL if r["id"] not in control_done]
t_missing = [r["id"] for r in HIST3_FULL if r["id"] not in treatment_done]
if c_missing or t_missing:
    print(f"WARNING: nohist missing {len(c_missing)}, hist3 missing {len(t_missing)} -- "
          f"re-run the full-run cell to finish before trusting the analysis below.")

c_scored = [r for r in CONTROL_FULL if r["id"] in control_done]
t_scored = [r for r in HIST3_FULL if r["id"] in treatment_done]
c_raws = [control_done[r["id"]] for r in c_scored]
t_raws = [treatment_done[r["id"]] for r in t_scored]

C = EVAL.score_predictions_normalized(c_scored, c_raws)
T = EVAL.score_predictions_normalized(t_scored, t_raws)

def banner(tag, M):
    lines = [f"=== {tag} (N={M['n_samples']}) ==="]
    lines.append(f"Accuracy/WA={M['accuracy_WA']:.2f}  UA={M['UA_unweighted_recall']:.2f}  "
                 f"macro-F1={M['macro_f1']:.2f}  weighted-F1={M['weighted_f1']:.2f}")
    lines.append(f"match tiers: {M['match_tier_counts']}  ambiguous_multi_label={M['n_ambiguous_multi_label']}")
    return "\n".join(lines)

print(banner("NoHist_s14", C))
print()
print(banner("Hist3_s14", T))

for tag, M in [(CONTROL_TAG, C), (TREATMENT_TAG, T)]:
    with open(os.path.join(OUTPUT_DIR, f"metrics_{tag}.json"), "w", encoding="utf-8") as f:
        json.dump({k: v for k, v in M.items() if k != "predictions"}, f, ensure_ascii=False, indent=2)
    with open(os.path.join(OUTPUT_DIR, f"predictions_{tag}.json"), "w", encoding="utf-8") as f:
        json.dump(M["predictions"], f, ensure_ascii=False, indent=2)
    pd.DataFrame(M["confusion_matrix"]["rows_gold_cols_pred"],
                 index=EVAL.IEMOCAP_LABELS, columns=EVAL.IEMOCAP_LABELS
                 ).to_csv(os.path.join(OUTPUT_DIR, f"confusion_{tag}.csv"))
make_zip_bundle("after scoring both arms")


=== NoHist_s14 (N=1200) ===
Accuracy/WA=43.17  UA=38.78  macro-F1=38.37  weighted-F1=41.01
match tiers: {'exact': 243, 'word_match': 957, 'edit_distance_fallback': 0}  ambiguous_multi_label=0

=== Hist3_s14 (N=1200) ===
Accuracy/WA=44.83  UA=41.02  macro-F1=40.91  weighted-F1=43.22
match tiers: {'exact': 369, 'word_match': 830, 'edit_distance_fallback': 1}  ambiguous_multi_label=0
[checkpoint zip] /kaggle/working/results_bundle.zip (0.04 MB) -- after scoring both arms -- download it now from Data > Output if you want an immediate local copy


'/kaggle/working/results_bundle.zip'

In [28]:
# ================= S1-4 CALIBRATION ANALYSIS (the point of this notebook) =================
# All six primary analyses requested. Nothing here reads or references Session 5.
if c_missing or t_missing:
    print("SKIPPING analysis -- finish the full run first (see WARNING above).")
else:
    nohist_by_id = {r["id"]: r for r in C["predictions"]}
    hist3_by_id  = {r["id"]: r for r in T["predictions"]}
    ids = [r["id"] for r in c_scored]
    N = len(ids)

    agree_ids = [i for i in ids if nohist_by_id[i]["normalized_prediction"] == hist3_by_id[i]["normalized_prediction"]]
    disagree_ids = [i for i in ids if i not in set(agree_ids)]

    def acc(idlist, by_id):
        if not idlist: return float("nan")
        return 100 * sum(1 for i in idlist if by_id[i]["normalized_prediction"] == by_id[i]["gold"]) / len(idlist)

    def macro_f1_of(idlist, by_id):
        if not idlist: return float("nan")
        eld, _ = EVAL.get_labels_attr("iemocap")
        from sklearn.metrics import f1_score
        g = [eld[by_id[i]["gold"]] for i in idlist]
        p = [eld[by_id[i]["normalized_prediction"]] for i in idlist]
        return f1_score(g, p, labels=list(range(6)), average="macro", zero_division=0) * 100

    print("="*78)
    print("1. Agreement rate")
    print("="*78)
    print(f"  agree: {len(agree_ids)}/{N} = {100*len(agree_ids)/N:.1f}%   "
          f"disagree: {len(disagree_ids)}/{N} = {100*len(disagree_ids)/N:.1f}%")

    print("\n" + "="*78)
    print("2. Accuracy / macro-F1 where they AGREE")
    print("="*78)
    print(f"  accuracy: {acc(agree_ids, nohist_by_id):.2f}%   macro-F1: {macro_f1_of(agree_ids, nohist_by_id):.2f}")

    print("\n" + "="*78)
    print("3. Accuracy of EACH ARM where they DISAGREE")
    print("="*78)
    print(f"  NoHist_s14 accuracy on disagreement subset: {acc(disagree_ids, nohist_by_id):.2f}%  (n={len(disagree_ids)})")
    print(f"  Hist3_s14  accuracy on disagreement subset: {acc(disagree_ids, hist3_by_id):.2f}%  (n={len(disagree_ids)})")

    print("\n" + "="*78)
    print("4. Does agreement significantly separate lower-risk from higher-risk cases?")
    print("="*78)
    from scipy import stats as _st
    n_agree_correct = sum(1 for i in agree_ids if nohist_by_id[i]["normalized_prediction"] == nohist_by_id[i]["gold"])
    n_dis_nohist_correct = sum(1 for i in disagree_ids if nohist_by_id[i]["normalized_prediction"] == nohist_by_id[i]["gold"])
    n_dis_hist3_correct = sum(1 for i in disagree_ids if hist3_by_id[i]["normalized_prediction"] == hist3_by_id[i]["gold"])
    table_a = [[n_agree_correct, len(agree_ids)-n_agree_correct],
               [n_dis_nohist_correct, len(disagree_ids)-n_dis_nohist_correct]]
    chi2, p_a, dof, exp = _st.chi2_contingency(table_a)
    print(f"  agree-accuracy vs NoHist-on-disagree-accuracy: chi2={chi2:.2f} p={p_a:.5f}")
    table_b = [[n_agree_correct, len(agree_ids)-n_agree_correct],
               [n_dis_hist3_correct, len(disagree_ids)-n_dis_hist3_correct]]
    chi2b, p_b, dofb, expb = _st.chi2_contingency(table_b)
    print(f"  agree-accuracy vs Hist3-on-disagree-accuracy:  chi2={chi2b:.2f} p={p_b:.5f}")
    print("  (NOTE: per the pre-registered power analysis, the SECOND comparison (vs Hist3-on-")
    print("   disagree) was expected to be underpowered at N=1200 -- report the p-value and CI")
    print("   honestly here, do not treat a non-significant result on this one as a failure of")
    print("   the whole calibration; the FIRST comparison and the McNemar test below are the")
    print("   well-powered primary tests.)")

    print("\n" + "="*78)
    print("5. Class-wise agreement / reliability, especially frustrated")
    print("="*78)
    for lab in EVAL.IEMOCAP_LABELS:
        cls_ids = [i for i in ids if nohist_by_id[i]["gold"] == lab]
        cls_agree = [i for i in cls_ids if i in set(agree_ids)]
        cls_disagree = [i for i in cls_ids if i not in set(agree_ids)]
        print(f"  {lab:<12} n={len(cls_ids):4d}  agree_rate={100*len(cls_agree)/max(len(cls_ids),1):5.1f}%  "
              f"acc|agree={acc(cls_agree, nohist_by_id):5.1f}%  "
              f"acc|disagree(NoHist)={acc(cls_disagree, nohist_by_id):5.1f}%  "
              f"acc|disagree(Hist3)={acc(cls_disagree, hist3_by_id):5.1f}%  (n_disagree={len(cls_disagree)})")

    print("\n" + "="*78)
    print("6. Secondary: paired NoHist_s14 vs Hist3_s14 performance (McNemar)")
    print("="*78)
    P = EVAL.paired_comparison(ids, C, ids, T)
    print(json.dumps({k: v for k, v in P.items() if k != "flips"}, indent=2))
    b, c_ = P["only_control_correct"], P["only_treatment_correct"]
    n_disc = b + c_
    if n_disc > 0:
        p_mcnemar = 2 * min(1.0, sum(_st.binom.pmf(k, n_disc, 0.5) for k in range(0, min(b, c_) + 1)))
        print(f"  McNemar exact p = {p_mcnemar:.5f}  (discordant pairs = {n_disc}; pre-registered target was >=134)")

    calib_summary = {
        "n_total": N, "n_agree": len(agree_ids), "n_disagree": len(disagree_ids),
        "agree_rate_pct": round(100*len(agree_ids)/N, 3),
        "accuracy_when_agree_pct": round(acc(agree_ids, nohist_by_id), 3),
        "macro_f1_when_agree": round(macro_f1_of(agree_ids, nohist_by_id), 3),
        "nohist_accuracy_when_disagree_pct": round(acc(disagree_ids, nohist_by_id), 3),
        "hist3_accuracy_when_disagree_pct": round(acc(disagree_ids, hist3_by_id), 3),
        "chi2_agree_vs_nohist_disagree_p": p_a, "chi2_agree_vs_hist3_disagree_p": p_b,
        "mcnemar_nohist_vs_hist3_p": p_mcnemar if n_disc > 0 else None,
        "paired_comparison": P,
    }
    with open(os.path.join(OUTPUT_DIR, "s14_calibration_summary.json"), "w", encoding="utf-8") as f:
        json.dump(calib_summary, f, ensure_ascii=False, indent=2)
    make_zip_bundle("after calibration analysis")
    print("\nwrote s14_calibration_summary.json")
    print("\nREMINDER: any fallback-rule decision must be made from THIS S1-4 output only.")
    print("Session 5 was not read, referenced, or used anywhere in this notebook.")


1. Agreement rate
  agree: 981/1200 = 81.8%   disagree: 219/1200 = 18.2%

2. Accuracy / macro-F1 where they AGREE
  accuracy: 46.89%   macro-F1: 41.75

3. Accuracy of EACH ARM where they DISAGREE
  NoHist_s14 accuracy on disagreement subset: 26.48%  (n=219)
  Hist3_s14  accuracy on disagreement subset: 35.62%  (n=219)

4. Does agreement significantly separate lower-risk from higher-risk cases?
  agree-accuracy vs NoHist-on-disagree-accuracy: chi2=29.56 p=0.00000
  agree-accuracy vs Hist3-on-disagree-accuracy:  chi2=8.75 p=0.00309
  (NOTE: per the pre-registered power analysis, the SECOND comparison (vs Hist3-on-
   disagree) was expected to be underpowered at N=1200 -- report the p-value and CI
   honestly here, do not treat a non-significant result on this one as a failure of
   the whole calibration; the FIRST comparison and the McNemar test below are the
   well-powered primary tests.)

5. Class-wise agreement / reliability, especially frustrated
  happy        n=  94  agree_rate= 8

In [29]:
# ================= FINAL PERSISTENCE CHECK -- do not skip this cell =================
def _line_count(path):
    if not os.path.exists(path):
        return None
    with open(path, encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())

CRITICAL_FILES = {
    "NoHist_s14 raw predictions (jsonl)": CONTROL_JSONL,
    "Hist3_s14 raw predictions (jsonl)": TREATMENT_JSONL,
    "S1-4 calibration summary (primary result)": os.path.join(OUTPUT_DIR, "s14_calibration_summary.json"),
}
EXPECTED_ROWS = len(CONTROL_FULL)

print("="*78)
print("FINAL PERSISTENCE CHECK")
print("="*78)
all_ok = True
for label, path in CRITICAL_FILES.items():
    exists = os.path.exists(path)
    size = os.path.getsize(path) if exists else 0
    if not exists:
        print(f"[MISSING] {label}: {path}"); all_ok = False
    elif size == 0:
        print(f"[EMPTY]   {label}: {path}"); all_ok = False
    else:
        if path.endswith(".jsonl"):
            n = _line_count(path)
            note = f"{n}/{EXPECTED_ROWS} rows" + ("" if n == EXPECTED_ROWS else "  <-- INCOMPLETE")
            if n != EXPECTED_ROWS: all_ok = False
            print(f"[ok]      {label}: {size/1e6:.2f} MB, {note}")
        else:
            print(f"[ok]      {label}: {size/1e3:.1f} KB")

zpath = make_zip_bundle("FINAL")
print(f"\nfull results/ listing: {sorted(os.listdir(OUTPUT_DIR))}")
print("\n" + "="*78)
print("ALL CRITICAL FILES PRESENT AND COMPLETE." if all_ok else
      "CRITICAL: one or more files above are MISSING/EMPTY/INCOMPLETE. Re-run the full-run "
      "cell (and the analysis cell) before closing this session.")
print("="*78)
print("""
NOW DO ONE OR BOTH BEFORE CLOSING THIS SESSION:
  1) IMMEDIATE: Data -> Output -> "results_bundle.zip" -> Download.
  2) DURABLE: "Save Version" -> "Save & Run All (Commit)" (NOT Quick Save); verify the
     committed Version's Output tab lists pred_s14_nohist_newparser.jsonl,
     pred_s14_hist3_newparser.jsonl, s14_calibration_summary.json, results_bundle.zip.
""")


FINAL PERSISTENCE CHECK
[ok]      NoHist_s14 raw predictions (jsonl): 0.12 MB, 1200/1200 rows
[ok]      Hist3_s14 raw predictions (jsonl): 0.13 MB, 1200/1200 rows
[ok]      S1-4 calibration summary (primary result): 14.4 KB
[checkpoint zip] /kaggle/working/results_bundle.zip (0.04 MB) -- FINAL -- download it now from Data > Output if you want an immediate local copy

full results/ listing: ['confusion_s14_hist3_newparser.csv', 'confusion_s14_nohist_newparser.csv', 'metrics_s14_hist3_newparser.json', 'metrics_s14_nohist_newparser.json', 'pred_s14_hist3_newparser.jsonl', 'pred_s14_nohist_newparser.jsonl', 'predictions_s14_hist3_newparser.json', 'predictions_s14_nohist_newparser.json', 's14_calibration_summary.json']

ALL CRITICAL FILES PRESENT AND COMPLETE.

NOW DO ONE OR BOTH BEFORE CLOSING THIS SESSION:
  1) IMMEDIATE: Data -> Output -> "results_bundle.zip" -> Download.
  2) DURABLE: "Save Version" -> "Save & Run All (Commit)" (NOT Quick Save); verify the
     committed Version's Outpu